In [ ]:
# XGBoost v2: eeg_id aggregation + GroupKFold + mu-law + KL metric
import os, sys, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold
from tqdm.notebook import tqdm

In [ ]:
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    META_PATH = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
else:
    DATA_ROOT = os.path.abspath('../')
    META_PATH = os.path.abspath('../data_meta_splits/train_test_split.csv')

EEG_DIR = os.path.join(DATA_ROOT, 'train_eegs')

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"EEG_DIR     : {EEG_DIR}")
print(f"META_PATH   : {META_PATH}")

In [ ]:
CFG = {
    "seed"                : 42,
    # data
    "fs"                  : 200,    # Hz
    "n_samples"           : 10_000, # 50 s × 200 Hz
    # XGBoost
    "max_depth"           : 6,
    "learning_rate"       : 0.05,
    "n_estimators"        : 2000,
    "subsample"           : 0.8,
    "colsample_bytree"    : 0.8,
    "min_child_weight"    : 5,
    "early_stopping_rounds": 50,
}

TARGETS     = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
CLASS_NAMES = ['seizure', 'lpd', 'gpd', 'lrda', 'grda', 'other']
EXPERT_TO_VOTE = {name: f'{name}_vote' for name in CLASS_NAMES}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(CFG)

In [ ]:
# ── channel definitions ──────────────────────────────────────────────────────
MONOPOLAR_COLS = ['Fp1', 'T3', 'O1', 'C3', 'Fp2', 'C4', 'T4', 'O2']

# 8 bipolar channels matching EEGDatasetWaveNet in data.py
BIPOLAR_PAIRS = [
    ('Fp1', 'T3'), ('T3', 'O1'),
    ('Fp1', 'C3'), ('C3', 'O1'),
    ('Fp2', 'C4'), ('C4', 'O2'),
    ('Fp2', 'T4'), ('T4', 'O2'),
]
BIPOLAR_NAMES = [f"{a}-{b}" for a, b in BIPOLAR_PAIRS]

# pre-compute FFT frequency-band masks once (fs=200, n=10000)
FS, N   = CFG["fs"], CFG["n_samples"]
_FREQS  = np.fft.rfftfreq(N, d=1.0 / FS)   # shape (5001,)
_DELTA  = (_FREQS >= 0.5)  & (_FREQS <  4.0)
_THETA  = (_FREQS >= 4.0)  & (_FREQS <  8.0)
_ALPHA  = (_FREQS >= 8.0)  & (_FREQS < 13.0)
_BETA   = (_FREQS >= 13.0) & (_FREQS < 30.0)

_COL_IDX = {c: i for i, c in enumerate(MONOPOLAR_COLS)}


def extract_eeg_features(parquet_path: str) -> np.ndarray:
    """88-dim feature vector for one EEG file (11 features × 8 bipolar channels)."""
    # --- load centre 50 s ---
    eeg = pd.read_parquet(parquet_path, columns=MONOPOLAR_COLS)
    rows = len(eeg)
    if rows < N:
        pad = N - rows
        top, bot = pad // 2, pad - pad // 2
        eeg = pd.concat(
            [eeg.iloc[:1].repeat(top), eeg, eeg.iloc[-1:].repeat(bot)],
            ignore_index=True,
        )
    offset = (len(eeg) - N) // 2
    data   = eeg.iloc[offset : offset + N].to_numpy(dtype=np.float32)  # (N, 8)
    data   = np.nan_to_num(data, nan=0.0)

    # --- mu-law encoding ---
    mu   = 256
    data = np.sign(data) * np.log(1 + mu * np.abs(data)) / np.log(1 + mu)

    # --- bipolar channels: vectorised, no Python loop over electrodes ---
    sig = np.stack(
        [data[:, _COL_IDX[a]] - data[:, _COL_IDX[b]] for a, b in BIPOLAR_PAIRS],
        axis=1,
    )  # (N, 8)

    # --- time-domain (7 × 8) ---
    td_mean   = sig.mean(axis=0)
    td_std    = sig.std(axis=0)
    td_min    = sig.min(axis=0)
    td_max    = sig.max(axis=0)
    td_skew   = skew(sig, axis=0)
    td_kurt   = kurtosis(sig, axis=0)
    td_energy = (sig ** 2).mean(axis=0)

    # --- band power (4 × 8) via rfft, vectorised over channels ---
    fft_power = (np.abs(np.fft.rfft(sig, axis=0)) ** 2) / N  # (N//2+1, 8)
    bp_delta  = fft_power[_DELTA].sum(axis=0)
    bp_theta  = fft_power[_THETA].sum(axis=0)
    bp_alpha  = fft_power[_ALPHA].sum(axis=0)
    bp_beta   = fft_power[_BETA ].sum(axis=0)

    # --- 11 × 8 = 88 features ---
    return np.concatenate([
        td_mean, td_std, td_min, td_max, td_skew, td_kurt, td_energy,
        bp_delta, bp_theta, bp_alpha, bp_beta,
    ])  # (88,)


def build_features(df: pd.DataFrame, eeg_dir: str, desc: str = ""):
    """Returns X (n,88), y_hard (n,), y_soft (n,6). Soft labels already normalised in df."""
    X, y_hard, y_soft = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        eeg_id = int(row['eeg_id'])
        feats  = extract_eeg_features(os.path.join(eeg_dir, f"{eeg_id}.parquet"))
        soft   = row[TARGETS].to_numpy(dtype=np.float32)
        X.append(feats)
        y_hard.append(int(np.argmax(soft)))
        y_soft.append(soft)
    return np.array(X, dtype=np.float32), np.array(y_hard), np.array(y_soft, dtype=np.float32)


# ── load metadata & aggregate by eeg_id ─────────────────────────────────────
meta = pd.read_csv(META_PATH)
meta = meta[meta['split'] == 'trainval'].reset_index(drop=True)

# sum votes per eeg_id; keep first patient_id and expert_consensus
agg = meta.groupby('eeg_id', as_index=False).agg(
    {**{t: 'sum' for t in TARGETS},
     'patient_id'        : 'first',
     'expert_consensus'  : 'first'}
)

# boost expert class by +10 before normalising
for idx, row in agg.iterrows():
    ec = str(row['expert_consensus']).lower()
    if ec in EXPERT_TO_VOTE:
        agg.at[idx, EXPERT_TO_VOTE[ec]] += 10

# normalise rows to sum=1
vote_sums = agg[TARGETS].sum(axis=1)
agg[TARGETS] = agg[TARGETS].div(vote_sums, axis=0)

print(f"Unique EEGs after aggregation: {len(agg):,}")

# ── GroupKFold by patient_id ─────────────────────────────────────────────────
groups  = agg['patient_id'].values
gkf     = GroupKFold(n_splits=5)
splits  = list(gkf.split(agg, groups=groups))
# fold 0: val; folds 1-4 collectively: train (single-fold run)
train_idx, val_idx = splits[0]
train_meta = agg.iloc[train_idx].reset_index(drop=True)
val_meta   = agg.iloc[val_idx  ].reset_index(drop=True)
print(f"Train EEGs: {len(train_meta):,} | Val EEGs: {len(val_meta):,}")

t0 = time.time()
X_train, y_train, y_train_soft = build_features(train_meta, EEG_DIR, desc="train")
X_val,   y_val,   y_val_soft   = build_features(val_meta,   EEG_DIR, desc="val  ")
print(f"\nX_train {X_train.shape} | X_val {X_val.shape}  [{time.time()-t0:.0f}s]")

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

params = {
    "objective"       : "multi:softprob",
    "num_class"       : 6,
    "eval_metric"     : "mlogloss",
    "tree_method"     : "hist",
    "device"          : "cuda",
    "max_depth"       : CFG["max_depth"],
    "learning_rate"   : CFG["learning_rate"],
    "subsample"       : CFG["subsample"],
    "colsample_bytree": CFG["colsample_bytree"],
    "min_child_weight": CFG["min_child_weight"],
    "seed"            : CFG["seed"],
}

evals_result = {}
model = xgb.train(
    params               = params,
    dtrain               = dtrain,
    num_boost_round      = CFG["n_estimators"],
    evals                = [(dtrain, "train"), (dval, "val")],
    early_stopping_rounds= CFG["early_stopping_rounds"],
    evals_result         = evals_result,
    verbose_eval         = 50,
)

print(f"\nBest iteration  : {model.best_iteration}")
print(f"Best val mlogloss: {model.best_score:.4f}")

In [ ]:
# ── metrics ──────────────────────────────────────────────────────────────────
prob_val  = model.predict(dval).reshape(-1, 6)          # (n, 6)  softprob output
pred_val  = prob_val.argmax(axis=1)

macro_f1  = f1_score(y_val, pred_val, average="macro", zero_division=0)
per_class = f1_score(y_val, pred_val, average=None,    zero_division=0)

# KL divergence against soft labels (Kaggle scoring convention)
kl_per_sample = (y_val_soft * np.log(np.clip(y_val_soft, 1e-7, 1) / np.clip(prob_val, 1e-7, 1))).sum(axis=1)
val_kl        = kl_per_sample.mean()

print(f"Val macro F1     : {macro_f1:.4f}")
print(f"Val KL divergence: {val_kl:.4f}")
print("\nPer-class F1:")
for name, f in zip(CLASS_NAMES, per_class):
    print(f"  {name:<10} {f:.4f}")

# ── plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

rounds = range(len(evals_result["train"]["mlogloss"]))
axes[0].plot(rounds, evals_result["train"]["mlogloss"], label="train")
axes[0].plot(rounds, evals_result["val"]["mlogloss"],   label="val")
axes[0].axvline(model.best_iteration, color="red", linestyle="--",
                label=f"best={model.best_iteration}")
axes[0].set_xlabel("Round"); axes[0].set_ylabel("mlogloss")
axes[0].set_title("XGBoost mlogloss"); axes[0].legend()

axes[1].bar(CLASS_NAMES, per_class, color="steelblue")
axes[1].axhline(macro_f1, color="red", linestyle="--",
                label=f"macro F1 = {macro_f1:.3f}")
axes[1].set_ylim(0, 1); axes[1].set_ylabel("F1")
axes[1].set_title("Per-class F1 (val)"); axes[1].legend()

plt.tight_layout()
plt.savefig("xgb_v2_eval.png", dpi=150)
plt.show()